# 06. Exploratory Data Analysis (EDA)

이 노트북에서는 `master_manifest_with_split.csv`를 기준으로
모델링 전 데이터 구조와 잠재적인 편향을 점검한다.

이번 EDA의 핵심 질문은 다음과 같다.

1. Train / Validation / Test가 의도한 비율로 분할되었는가?
2. REAL / FAKE 비율은 얼마나 불균형한가?
3. 장르(Electronic / Rock / Pop)는 split별로 비슷하게 유지되는가?
4. 각 AI generator는 split별로 어떻게 분포하는가?
5. generator별로 포함하는 `original_audio` 수가 다른가?
6. 특정 `original_audio`에 FAKE 샘플이 과도하게 많이 몰려 있는가?

> 이번 단계에서는 MFCC, Log-Mel 등 음향 특징을 아직 추출하지 않는다.
> 실제 음향 특징 EDA는 10초 segment 생성 이후 수행하는 것이 더 효율적이다.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

MANIFEST_PATH = PROJECT_ROOT / "data/metadata/master_manifest_with_split.csv"
EDA_DIR = PROJECT_ROOT / "results/eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

print("MANIFEST_PATH:", MANIFEST_PATH)
print("EDA_DIR      :", EDA_DIR)


## 1. 데이터 로드 및 기본 구조 확인

먼저 전체 행 수, `original_audio` 수, label, split, genre, generator 구조를 확인한다.


In [ ]:
df = pd.read_csv(MANIFEST_PATH)

print("===== BASIC STRUCTURE =====")
print("Rows                 :", len(df))
print("Unique original_audio:", df["original_audio"].nunique())
print("Columns              :", df.columns.tolist())

print("\nSplit values:")
print(df["split"].value_counts())

print("\nLabel values:")
print(df["label"].value_counts())

print("\nGenre values:")
print(df["genre"].value_counts())

print("\nFAKE generators:")
print(
    df.loc[df["label"] == "FAKE", "generator"]
    .value_counts()
)

display(df.head())


## 2. Split별 Group 수 확인

분할은 sample이 아니라 `original_audio` group 단위로 수행했기 때문에,
각 split에 몇 개의 source family가 배정되었는지 다시 확인한다.


In [ ]:
group_split_summary = (
    df[["original_audio", "genre", "split"]]
    .drop_duplicates("original_audio")
    .groupby("split")
    .agg(
        original_audio_count=("original_audio", "nunique")
    )
)

group_split_summary["ratio"] = (
    group_split_summary["original_audio_count"]
    / group_split_summary["original_audio_count"].sum()
)

display(group_split_summary)

ax = group_split_summary["original_audio_count"].plot(
    kind="bar",
    title="Number of Original Audio Groups by Split",
    rot=0
)
ax.set_xlabel("Split")
ax.set_ylabel("Number of original_audio groups")
plt.tight_layout()
plt.show()


## 3. Sample-level Split 분포

`original_audio`별 FAKE 생성물 수가 다르기 때문에,
group 비율이 70/15/15에 가깝더라도 sample-level 비율은 정확히 같지 않을 수 있다.


In [ ]:
sample_split_summary = (
    df["split"]
    .value_counts()
    .rename("sample_count")
    .to_frame()
)

sample_split_summary["ratio"] = (
    sample_split_summary["sample_count"]
    / sample_split_summary["sample_count"].sum()
)

display(sample_split_summary)

ax = sample_split_summary["sample_count"].plot(
    kind="bar",
    title="Number of Samples by Split",
    rot=0
)
ax.set_xlabel("Split")
ax.set_ylabel("Samples")
plt.tight_layout()
plt.show()


## 4. REAL / FAKE Class Imbalance 확인

현재 데이터는 원곡마다 REAL은 1개지만,
같은 원곡으로부터 여러 AI 생성물이 존재하므로 FAKE 수가 훨씬 많다.

이 비율은 이후 Logistic Regression, SVM, CNN 학습 시
class weight, sampling 또는 loss 설정을 결정하는 데 중요하다.


In [ ]:
label_summary = (
    df["label"]
    .value_counts()
    .rename("count")
    .to_frame()
)

label_summary["ratio"] = (
    label_summary["count"]
    / label_summary["count"].sum()
)

display(label_summary)

label_by_split = pd.crosstab(
    df["split"],
    df["label"]
)

label_ratio_by_split = pd.crosstab(
    df["split"],
    df["label"],
    normalize="index"
).round(4)

print("===== LABEL COUNT BY SPLIT =====")
display(label_by_split)

print("===== LABEL RATIO BY SPLIT =====")
display(label_ratio_by_split)

ax = label_by_split.plot(
    kind="bar",
    title="REAL / FAKE Distribution by Split",
    rot=0
)
ax.set_xlabel("Split")
ax.set_ylabel("Samples")
plt.tight_layout()
plt.show()


## 5. Group-level Genre 분포

장르 stratification은 `original_audio` group 기준으로 수행했다.
따라서 먼저 group-level 장르 분포를 확인한다.


In [ ]:
group_level = (
    df[["original_audio", "genre", "split"]]
    .drop_duplicates("original_audio")
    .copy()
)

group_genre_count = pd.crosstab(
    group_level["split"],
    group_level["genre"]
)

group_genre_ratio = pd.crosstab(
    group_level["split"],
    group_level["genre"],
    normalize="index"
).round(4)

print("===== GROUP-LEVEL GENRE COUNT =====")
display(group_genre_count)

print("===== GROUP-LEVEL GENRE RATIO =====")
display(group_genre_ratio)

ax = group_genre_count.plot(
    kind="bar",
    title="Genre Distribution by Split (Group Level)",
    rot=0
)
ax.set_xlabel("Split")
ax.set_ylabel("Original audio groups")
plt.tight_layout()
plt.show()


## 6. Sample-level Genre 분포

FAKE 생성물 개수가 원곡마다 다르기 때문에
sample-level 장르 분포는 group-level 분포와 조금 달라질 수 있다.


In [ ]:
sample_genre_count = pd.crosstab(
    df["split"],
    df["genre"]
)

sample_genre_ratio = pd.crosstab(
    df["split"],
    df["genre"],
    normalize="index"
).round(4)

print("===== SAMPLE-LEVEL GENRE COUNT =====")
display(sample_genre_count)

print("===== SAMPLE-LEVEL GENRE RATIO =====")
display(sample_genre_ratio)

ax = sample_genre_count.plot(
    kind="bar",
    title="Genre Distribution by Split (Sample Level)",
    rot=0
)
ax.set_xlabel("Split")
ax.set_ylabel("Samples")
plt.tight_layout()
plt.show()


## 7. AI Generator 전체 분포

REAL의 `generator`는 결측값이므로 제외하고 FAKE만 분석한다.

generator별 총 샘플 수가 크게 다르면,
모델이 특정 generator의 특징을 과도하게 학습할 가능성이 있다.


In [ ]:
fake = df[df["label"] == "FAKE"].copy()

generator_summary = (
    fake["generator"]
    .value_counts()
    .rename("sample_count")
    .to_frame()
)

generator_summary["ratio"] = (
    generator_summary["sample_count"]
    / generator_summary["sample_count"].sum()
)

display(generator_summary)

ax = generator_summary["sample_count"].sort_values().plot(
    kind="barh",
    title="FAKE Samples by Generator"
)
ax.set_xlabel("Samples")
ax.set_ylabel("Generator")
plt.tight_layout()
plt.show()


## 8. Split별 Generator 분포

각 AI generator가 Train / Validation / Test에 어떻게 배정되었는지 확인한다.

이번 기본 split은 generator holdout 실험이 아니라
`original_audio` 기준 in-domain split이므로,
대부분의 generator가 모든 split에 존재할 수 있다.


In [ ]:
generator_by_split = pd.crosstab(
    fake["generator"],
    fake["split"]
)

display(generator_by_split)

generator_ratio_by_split = pd.crosstab(
    fake["generator"],
    fake["split"],
    normalize="index"
).round(4)

print("===== GENERATOR SPLIT RATIO =====")
display(generator_ratio_by_split)

ax = generator_by_split.plot(
    kind="bar",
    title="Generator Distribution across Splits"
)
ax.set_xlabel("Generator")
ax.set_ylabel("Samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Generator별 original_audio Coverage

generator마다 전체 296개 원곡을 모두 생성한 것은 아니다.

따라서 단순 샘플 수뿐 아니라
각 generator가 몇 개의 서로 다른 `original_audio`를 포함하는지도 확인한다.


In [ ]:
generator_coverage = (
    fake
    .groupby("generator")
    .agg(
        samples=("sample_id", "size"),
        unique_original_audio=("original_audio", "nunique"),
    )
    .sort_values(
        ["unique_original_audio", "samples"],
        ascending=False
    )
)

generator_coverage["coverage_ratio"] = (
    generator_coverage["unique_original_audio"] / 296
).round(4)

display(generator_coverage)

ax = generator_coverage["unique_original_audio"].sort_values().plot(
    kind="barh",
    title="Original Audio Coverage by Generator"
)
ax.set_xlabel("Unique original_audio")
ax.set_ylabel("Generator")
plt.tight_layout()
plt.show()


## 10. Generator × Genre 분포

특정 generator가 특정 장르에만 과도하게 집중되어 있는지 확인한다.

generator와 genre가 강하게 연관되어 있다면
모델이 AI artifact가 아니라 장르 특성을 이용할 위험이 있으므로 참고해야 한다.


In [ ]:
generator_genre = pd.crosstab(
    fake["generator"],
    fake["genre"]
)

generator_genre_ratio = pd.crosstab(
    fake["generator"],
    fake["genre"],
    normalize="index"
).round(4)

print("===== GENERATOR × GENRE COUNT =====")
display(generator_genre)

print("===== GENERATOR × GENRE RATIO =====")
display(generator_genre_ratio)

ax = generator_genre.plot(
    kind="bar",
    title="Genre Distribution within Each Generator"
)
ax.set_xlabel("Generator")
ax.set_ylabel("Samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 11. original_audio별 Sample 수 분포

원곡마다 연결된 AI 생성물 수가 다르므로,
특정 source family가 데이터에서 과도하게 큰 비중을 차지하는지 확인한다.


In [ ]:
group_sample_count = (
    df
    .groupby(["original_audio", "split", "genre"])
    .agg(
        total_samples=("sample_id", "size"),
        fake_samples=("label", lambda x: (x == "FAKE").sum()),
    )
    .reset_index()
)

print("===== TOTAL SAMPLES PER ORIGINAL_AUDIO =====")
display(group_sample_count["total_samples"].describe())

print("===== FAKE SAMPLES PER ORIGINAL_AUDIO =====")
display(group_sample_count["fake_samples"].describe())

print("\nLargest source families:")
display(
    group_sample_count
    .sort_values("total_samples", ascending=False)
    .head(20)
)

ax = group_sample_count["fake_samples"].plot(
    kind="hist",
    bins=15,
    title="Distribution of FAKE Samples per Original Audio"
)
ax.set_xlabel("FAKE samples per original_audio")
plt.tight_layout()
plt.show()


## 12. Split별 source-family 크기 비교

Train / Validation / Test 중 특정 split에 큰 source family가 과도하게 몰렸는지 확인한다.


In [ ]:
family_size_by_split = (
    group_sample_count
    .groupby("split")["fake_samples"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(2)
)

display(family_size_by_split)


## 13. EDA 요약 테이블 저장

추후 보고서와 발표 자료에서 재사용할 수 있도록
핵심 요약표를 `results/eda/`에 CSV로 저장한다.


In [ ]:
group_split_summary.to_csv(
    EDA_DIR / "group_split_summary.csv",
    encoding="utf-8-sig"
)

sample_split_summary.to_csv(
    EDA_DIR / "sample_split_summary.csv",
    encoding="utf-8-sig"
)

label_by_split.to_csv(
    EDA_DIR / "label_by_split.csv",
    encoding="utf-8-sig"
)

group_genre_count.to_csv(
    EDA_DIR / "group_genre_by_split.csv",
    encoding="utf-8-sig"
)

sample_genre_count.to_csv(
    EDA_DIR / "sample_genre_by_split.csv",
    encoding="utf-8-sig"
)

generator_summary.to_csv(
    EDA_DIR / "generator_summary.csv",
    encoding="utf-8-sig"
)

generator_by_split.to_csv(
    EDA_DIR / "generator_by_split.csv",
    encoding="utf-8-sig"
)

generator_coverage.to_csv(
    EDA_DIR / "generator_coverage.csv",
    encoding="utf-8-sig"
)

generator_genre.to_csv(
    EDA_DIR / "generator_genre.csv",
    encoding="utf-8-sig"
)

group_sample_count.to_csv(
    EDA_DIR / "original_audio_sample_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved EDA tables to:", EDA_DIR)


## 14. 핵심 EDA QC 요약

모델링 전 반드시 확인할 최소 조건을 요약한다.


In [ ]:
eda_qc = pd.DataFrame({
    "check": [
        "total_samples",
        "original_audio_groups",
        "real_samples",
        "fake_samples",
        "ai_generators",
        "train_groups",
        "val_groups",
        "test_groups",
        "missing_split",
        "missing_audio_path",
    ],
    "value": [
        len(df),
        df["original_audio"].nunique(),
        int((df["label"] == "REAL").sum()),
        int((df["label"] == "FAKE").sum()),
        fake["generator"].nunique(),
        group_level.loc[group_level["split"] == "train", "original_audio"].nunique(),
        group_level.loc[group_level["split"] == "val", "original_audio"].nunique(),
        group_level.loc[group_level["split"] == "test", "original_audio"].nunique(),
        int(df["split"].isna().sum()),
        int(df["audio_path"].isna().sum()),
    ]
})

display(eda_qc)

eda_qc_pass = (
    len(df) == 3458
    and df["original_audio"].nunique() == 296
    and int((df["label"] == "REAL").sum()) == 296
    and int((df["label"] == "FAKE").sum()) == 3162
    and fake["generator"].nunique() == 12
    and int(df["split"].isna().sum()) == 0
    and int(df["audio_path"].isna().sum()) == 0
)

print("===== FINAL RESULT =====")
print("EDA Core QC PASS:", eda_qc_pass)


## 다음 단계

EDA가 완료되면 다음 단계는 **10초 segment manifest 생성**이다.

예정 규칙:

```text
원곡 길이 >= 30초 → start / middle / end, 최대 3개
20~30초          → 2개
10~20초          → 1개
10초 미만        → 별도 검토
```

분할은 이미 완료되어 있으므로 segment를 만든 뒤에도
각 segment는 반드시 원본 track의 기존 `split`을 그대로 상속한다.

```text
master_manifest_with_split.csv
        ↓
10초 segment manifest
        ↓
Handcrafted Features
        ↓
Logistic Regression / SVM
        ↓
Log-Mel CNN
```
